# Types of streams

In stream processing, we distinguish between two types of streams:
* [*append-only streams*](#append_only)
* [*changelog streams*](#changelog)

This chapter is about how to handle these two types of streams with Kafi Streams and how seamlessly combine the two in a common Kafi Streams topology ([Combining append-only and changelog streams](#combining)).


## Overview

[Preparation](#prep)

* [Append-only streams](#append_only)
* [Changelog streams](#changelog)
* [Combining append-only and changelog streams](#combining)


---
<a id="prep"></a>
## Preparation

Before we start off, we first prepare for the examples to follow:

In [7]:
import sys
sys.path.insert(1, ".")
sys.path.insert(1, "../..")

from kafi.streams.topologynode import TopologyNode as Tn

from generators import ClickGenerator, CustomerGenerator
click_generator = ClickGenerator()
customer_generator = CustomerGenerator()

click_source_str = "clicks"
customer_source_str = "customers"

sink_str = "sink"


---
<a id="append_only"></a>
## Append-only streams

*Append-only streams* are typically streams of *transactional data*. We already saw an example of this in the [Quickstart](quickstart.ipynb) (and in many examples thereafter): clicks on a web site/web shop.

And we already learnt in [Time operators](operators/time.ipynb) how to deal with them:
* either by using the `expire()` operator to just time out old input records,
* or by using time windows.

This is basically all you have to know about how to handle append-only streams in Kafi Streams :-)

---
<a id="changelog"></a>
## Changelog streams

*Changelog streams* differ from *append-only streams* in that they typically deal with *master data*.

This type of stream often comes in from *Change Data Capture* (*CDC*) systems like *Debezium*. Or, in other circumstances, from compacted Kafka topics which might also bear so-called *tombstone messages*.

Changelog streams are typically not timed out in a stream processor because the volume of the master data tables in the source databases stays relatively stable. This kind of stream is thus more about *updates* and *deletes*, e.g. of customer data, as we already saw in [Quickstart](quickstart.ipynb).

Now how can you handle changelog streams in Kafi Streams?

There are two basic mechanisms:
1. If you get CDC-based changelog streams, e.g. in Debezium format, the only thing you have to do is to adapt the way these streams are converted into Kafi Streams"/pydbsp"s *Z-sets* using the stateless `to_zSet()` operator at the source.
2. If you have compacted Kafka topics, you can use the stateful `compact()` operator.

The big upside of using CDC-based changelog streams is that the conversion from the input stream into Kafi Streams' Z-sets does not require any state (`to_zSet` is always stateless). The `compact()` operator does have state because it has to remember the last value per key.

Hence, if you can influence the decision for how the upstream data comes into Kafi Streams, CDC-based streams are always the preferred choice.

<a id="to_zset"></a>
### to_zSet()

Currently the only CDC-source supported is Debezium. You can convert the source stream from Debezium format to Kafi Streams' *Z-sets* using `to_zSet(Tn.from_debezium)`.

Here is an example topology.

In [6]:
sink_tn = (
    Tn.source(customer_source_str)
    ###
    # to_zSet() operator configured to handle CDC input in Debezium format 
    ###
    .to_zSet(Tn.from_debezium)
    #
    .group_by_count(
        key_fun=lambda r: r["value"]["id"],
        project_fun=lambda key_any, agg_any: {"id": key_any, "count": agg_any})
)

tn = Tn.build(sink_tn)
tn.from_zSet(Tn._to_records)

print("Step 1")

customer_input_m_list = [{"key": "42", "value": {"op": "c",
                                                 "after": {"id": 42, "name": "Betty Graham MD"}}},
                         {"key": "4711", "value": {"op": "c",
                                                   "after": {"id": 4711, "name": "Frank Frank"}}}]
print("\nInput (customers):")
for m in customer_input_m_list:
    print(m)

output_m_list = tn.process({customer_source_str: customer_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)

print("\n---")
print("\nStep 2")

customer_input_m_list = [{"key": "42",
                          "value": {"op": "u",
                                    "before": {"id": 42, "name": "Betty Graham MD"},
                                    "after": {"id": 42, "name": "Betty G. Miller"}}}]
print("\nInput (customers):")
for m in customer_input_m_list:
    print(m)

output_m_w_tuple_list = tn.process({customer_source_str: customer_input_m_list})
print("\nOutput:")
for m_w_tuple in output_m_w_tuple_list:
    print(m_w_tuple)

print("\n---")
print("\nStep 3")

customer_input_m_list = [{"key": "42", "value": {"op": "d",
                                                 "before": {"id": 42, "name": "Betty G. Miller"}}}]
print("\nInput (customers):")
for m_w_tuple in customer_input_m_list:
    print(m_w_tuple)

output_m_w_tuple_list = tn.process({customer_source_str: customer_input_m_list})
print("\nOutput:")
for m_w_tuple in output_m_w_tuple_list:
    print(m_w_tuple)

print("\n---")
print("\nStep 4")

customer_input_m_list = [{"key": "42", "value": {"op": "c",
                                                 "after": {"id": 42, "name": "Betty G. Miller"}}}]
print("\nInput (customers):")
for m_w_tuple in customer_input_m_list:
    print(m_w_tuple)

output_m_w_tuple_list = tn.process({customer_source_str: customer_input_m_list})
print("\nOutput:")
for m_w_tuple in output_m_w_tuple_list:
    print(m_w_tuple)


Step 1

Input (customers):
{'key': '42', 'value': {'op': 'c', 'after': {'id': 42, 'name': 'Betty Graham MD'}}}
{'key': '4711', 'value': {'op': 'c', 'after': {'id': 4711, 'name': 'Frank Frank'}}}

Output:
({'id': 42, 'count': 1}, 1)
({'id': 4711, 'count': 1}, 1)

---

Step 2

Input (customers):
{'key': '42', 'value': {'op': 'u', 'before': {'id': 42, 'name': 'Betty Graham MD'}, 'after': {'id': 42, 'name': 'Betty G. Miller'}}}

Output:

---

Step 3

Input (customers):
{'key': '42', 'value': {'op': 'd', 'before': {'id': 42, 'name': 'Betty G. Miller'}}}

Output:
({'id': 42, 'count': 1}, -1)

---

Step 4

Input (customers):
{'key': '42', 'value': {'op': 'c', 'after': {'id': 42, 'name': 'Betty G. Miller'}}}

Output:
({'id': 42, 'count': 1}, 1)


In the example, we go through four steps:
1. We insert two customers. Both then have `count = 1`.
2. We update customer `42`. The counts stay unchanged, hence the output is empty.
3. We delete customer `42`. The count for this customer is retracted (=weight `-1`).
4. We insert customer `42` once again. Its count goes up to `1` again.


<a id="compact"></a>
### compact()

If you don't have the luxury of fully-fledged CDC changelog streams coming in but only Kafka-compacted topic-style changelog streams (maybe also including tombstone messages), the `compact()` operator comes to your rescue.

Here is the example from the [pydbsp integration](pydbsp.ipynb) chapter once again:

In [ ]:
sink_tn = (
    Tn.source(customer_source_str)
    ###
    # compact() operator - compact changelog streams by key
    ###
    .compact()
    #
    .group_by_count(
        key_fun=lambda r: r["value"]["id"],
        project_fun=lambda key_any, agg_any: {"id": key_any, "count": agg_any})
)

tn = Tn.build(sink_tn)
tn.from_zSet(Tn._to_records)

print("Step 1")

customer_input_m_list = [{"key": "42", "value": {"id": 42, "name": "Betty Graham MD"}},
                         {"key": "4711", "value": {"id": 4711, "name": "Frank Frank"}}]
print("\nInput (customers):")
for m in customer_input_m_list:
    print(m)

output_m_list = tn.process({customer_source_str: customer_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)

print("\n---")
print("\nStep 2")

customer_input_m_list = [{"key": "42", "value": {"id": 42, "name": "Betty G. Miller"}}]
print("\nInput (customers):")
for m in customer_input_m_list:
    print(m)

output_m_w_tuple_list = tn.process({customer_source_str: customer_input_m_list})
print("\nOutput:")
for m_w_tuple in output_m_w_tuple_list:
    print(m_w_tuple)

print("\n---")
print("\nStep 3")

customer_input_m_list = [{"key": "42", "value": None}]
print("\nInput (customers):")
for m_w_tuple in customer_input_m_list:
    print(m_w_tuple)

output_m_w_tuple_list = tn.process({customer_source_str: customer_input_m_list})
print("\nOutput:")
for m_w_tuple in output_m_w_tuple_list:
    print(m_w_tuple)

print("\n---")
print("\nStep 4")

customer_input_m_list = [{"key": "42", "value": {"id": 42, "name": "Betty G. Miller"}}]
print("\nInput (customers):")
for m_w_tuple in customer_input_m_list:
    print(m_w_tuple)

output_m_w_tuple_list = tn.process({customer_source_str: customer_input_m_list})
print("\nOutput:")
for m_w_tuple in output_m_w_tuple_list:
    print(m_w_tuple)


As you can see, the example proceeds in exactly the same way as the one before for CDC-based changelog streams in Debezium format:
1. We insert two customers. Both then have `count = 1`.
2. We update customer `42`. The counts stay unchanged, hence the output is empty.
3. We delete customer `42`. The count for this customer is retracted (=weight `-1`).
4. We insert customer `42` once again. Its count goes up to `1` again.

The difference is that the input stream is a typical Kafka-compacted-topic-style changelog stream where the information about the previous value of a record with a certain key (carried in the `before` sub record in Debezium) is missing.

Here, the `compact()` operator serves to recover the previous values from its state.

---
<a id="combining"></a>
## Combining append-only and changelog streams

One of the cool features in Kafi Streams is that append-only streams and changelog streams can be seamlessly combined. There is no distinction between `KStream` and `KTable` as in Kafka Streams, for instance. Both kinds of streams just boil down to Z-sets at the source.

In fact, if, aside from tombstone handling (where the `compact()` operator is essential), and of course memory considerations, you could also just ignore the difference between the two types of streams, as we did in our very first example topology in [Quickstart](quickstart.ipynb).

Let's revisit this first topology and augment it with the means of Kafi Streams to handle the two kinds of streams coming in:
* the clicks (append-only transactional data), and
* the customers (changelog master data)

In [17]:
# 1. Boilerplate

import sys
sys.path.insert(1, "../..")

from kafi.streams.streams import Streams

import logging
logging.basicConfig(level=logging.INFO)

# 2. Connect to Kafka

from kafi.kafka.cluster.cluster import Cluster
c = Cluster({"kafka": {"bootstrap.servers": "localhost:9092"}})

# 3. Specify the Topology

click_source_str = "clicks"
customer_source_str = "customers"
sink_str = "joined"

## a) Clicks

click_tn = (
    Streams.source(c, click_source_str)
    ###
    # expire append-only data
    ###
    .expire(
        ts_fun=lambda r: r["value"]["ts"],
        expiry_fun=lambda ts: ts + click_generator.ts_step_int * 1000)
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"], "ts": r["value"]["ts"]})
    .filter(lambda r: r["view_time"] > 20)
    .distinct()
)

## b) Customers

customer_tn = (
    Streams.source(c, customer_source_str)
    ###
    # compact Kafka changelog data
    ###
    # .compact()
    .map(lambda r: {"id": r["value"]["id"], "name": r["value"]["name"]})
    .distinct()
)

## c) Join and Sink

sink_tn = (
    click_tn
    .join(
        customer_tn,
        lambda l_r: l_r["customer_id"],
        lambda r_r: r_r["id"],
        lambda l_r, r_r: {"value": {
            "customer_id": l_r["customer_id"],
            "view_time": l_r["view_time"],
            "ts": l_r["ts"],
            "name": r_r["name"]}})
    .sink(c, sink_str)
)

# 4. Build the Topology

tn = Streams.build(sink_tn)


Let's start the Streams thread and feed it with example data:

In [19]:
tn.reset()

c.recreate(click_source_str)
c.recreate(customer_source_str)
c.recreate(sink_str)

stop = Streams.start_streams(tn, progress=True)

#

click_pr = c.producer(click_source_str)
customer_pr = c.producer(customer_source_str)

for i in range(100):
    click_m_dict_list = click_generator.generate(100)
    click_pr.produce_list(click_m_dict_list)

    customer_m_dict_list = customer_generator.generate(100)
    customer_pr.produce_list(customer_m_dict_list)

click_pr.close()
customer_pr.close()

INFO:kafi.streams.streams:Starting Streams...


(['customers'], 'streams_1788095073205')
(['clicks'], 'streams_1788095073205')


'customers'

Uptime: 5.153s, State size: 1787.41 KB, Source offsets: {'customers': {0: 10000}, 'clicks': {0: 10000}}, Sink outputs: 9011

And then stop it:

In [20]:
stop()
Streams.threads()

INFO:kafi.streams.streams:Safely stopping Streams...


Uptime: 10.025s, State size: 1069.47 KB, Source offsets: {'customers': {0: 10000}, 'clicks': {0: 10000}}, Sink outputs: 9011

INFO:kafi.streams.streams:...done.


[]

We get a similar output than before in the [Quickstart](quickstart.ipynb) chapter, but we have handled the two types of input streams in a more robust way:
* we let the `clicks` expire after `1000` steps, and
* we used `compact()` to properly handle the updates coming from the Kafka-compacted-topic-style changelog stream `customers`
